# Basic IMR Chart

The Individual-Moving Range (IMR) chart is the most versatile process behavior chart. It works with any data structure and is ideal for:

- Single measurements over time
- Small sample sizes (n=1)
- Time series monitoring

## What You'll Learn

1. Create an IMR chart for time series data
2. Interpret the Individual (I) and Moving Range (R) charts
3. Detect signals using Western Electric rules
4. Customize the visualization

## Setup

In [ ]:
import numpy as np
import pandas as pd
from processbehavior import ProcessDataFrame

## Create Sample Data

Let's simulate daily temperature readings from a manufacturing process. We'll include:
- Normal variation around a target of 72°F
- A special cause event (equipment malfunction) on day 25

In [ ]:
np.random.seed(42)

# 30 days of temperature readings
n_days = 30
temperatures = np.random.normal(72, 1.5, n_days)

# Add a special cause on day 25 (equipment malfunction)
temperatures[24] = 78.5  # Spike

# Create DataFrame
df = pd.DataFrame({
    'day': range(1, n_days + 1),
    'temperature': np.round(temperatures, 1)
})

print(f"Dataset: {len(df)} observations")
df.head(10)

## Create the ProcessDataFrame

Wrap your pandas DataFrame to enable the fluent API:

In [ ]:
pdf = ProcessDataFrame(df)

# IDE auto-completion for column names
print("Available columns:")
print(f"  - {pdf.columns.day}")
print(f"  - {pdf.columns.temperature}")

## Formulate the Study

For a simple time series, specify only the response and time variables.
No factors means this is SDS 4 (single stream over time).

In [ ]:
study = pdf.formulate(
    response=pdf.columns.temperature,
    time=pdf.columns.day
)

print(f"SDS: {study.sds} ({study.sds_name})")
print(f"Valid charts: {study.valid_charts}")
print(f"Recommended: {study.recommended_chart}")

## Analyze with IMR Chart

The IMR chart creates two linked charts:
- **I (Individual)**: Plots each observation against control limits
- **R (Moving Range)**: Plots the absolute difference between consecutive observations

In [ ]:
result = study.analyze()

print(f"Charts created: {result.all_charts}")

## View Chart Data

All results are plain pandas DataFrames:

In [ ]:
# Individual chart data
imr_data = result.get_chart('Imr')
print("IMR Chart Data:")
imr_data.head(10)

In [ ]:
# Control limit statistics
stats = result.get_statistics('Imr')
print("\nIMR Statistics:")
stats

## Visualize the Chart

Create an interactive Plotly chart:

In [ ]:
fig = result.plot()
fig.show()

## Enhanced Visualization

Add zone shading and statistics:

In [ ]:
fig = result.plot(
    show_zones=True,    # Show 1σ, 2σ, 3σ zones
    show_stats=True,    # Display statistics box
    show_signals=True   # Highlight out-of-control points
)
fig.show()

## Detect Signals

Apply Western Electric rules to find out-of-control conditions.

For IMR charts, all 8 rules can be applied:

In [ ]:
# Standard rules (1-4)
signals = result.detect_signals(chart='Imr', rules='standard')

print(f"Signals found: {signals.count}")
print(f"Has signals: {signals.has_signals}")

if signals.has_signals:
    print("\nViolations:")
    display(signals.violations)

In [ ]:
# Extended rules (1-8) for more sensitive detection
signals_extended = result.detect_signals(chart='Imr', rules='extended')

print(f"Extended rules found {signals_extended.count} signals")
if signals_extended.has_signals:
    print("\nSummary by rule:")
    print(signals_extended.summary)

## Plot with Rule Violations

Show all rule violations on the chart:

In [ ]:
fig = result.plot(
    show_zones=True,
    show_rules=True,  # Show all WECO rule violations
    template='processbehavior'
)
fig.show()

## Interpreting IMR Charts

### The Individual (I) Chart

- **Centerline (CL)**: The average of all observations
- **Upper Control Limit (UCL)**: CL + 2.66 × Average Moving Range
- **Lower Control Limit (LCL)**: CL - 2.66 × Average Moving Range

Points beyond the limits indicate **special cause variation**.

### The Moving Range (R) Chart

- **Centerline**: Average moving range
- **UCL**: 3.27 × Average Moving Range
- **LCL**: 0 (moving range cannot be negative)

Large moving ranges indicate **unstable process variation**.

### Reading the Charts Together

1. First check the R chart for stability
2. If R is stable, interpret the I chart
3. If R is unstable, address variation first

## Summary

In this tutorial, you learned:

- IMR charts work with individual observations (n=1)
- The I chart monitors the process level
- The R chart monitors process variation
- All 8 Western Electric rules apply to IMR charts
- Signals indicate special cause variation requiring investigation

## Next Steps

- {doc}`xbar-s-analysis` - Charts for subgrouped data
- {doc}`stratified-analysis` - Multiple streams analysis
- {doc}`signal-detection` - Deep dive into WECO rules